# 05 — What the model is doing, and where it fails

The baseline already gets 246 of 293. The 47 it misses are where the state
stopped behaving uniformly, and they're the interesting part.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.features import load_seat_features
from src.models import statewide_swing
from src.explain import (
    baseline_misses, local_swing_spread, logistic_coefficients,
    miss_summary, permutation_table,
)

seats = load_seat_features()
swing = statewide_swing(seats)

## Uniform swing isn't uniform

The baseline's one assumption is that every seat moves by the same amount.
Here is how badly that holds.

In [ ]:
spread = local_swing_spread(seats)
print(f"statewide swing      {swing:.2f}")
print(f"seat mean            {spread['local_swing'].mean():.2f}")
print(f"standard deviation   {spread['local_swing'].std():.2f}")
print(f"range                {spread['local_swing'].min():.1f} to {spread['local_swing'].max():.1f}")

A standard deviation of 5 points around a swing of 7.6. Some seats moved to
TMC while the state moved hard the other way.

![local swing](../figures/01_local_swing_spread.png)

## Where the misses are

This is the result I didn't expect. The baseline is *best* on the closest
seats and *worst* in the middle.

In [ ]:
by_region, by_band = miss_summary(seats)
by_band

Seats decided by under 5 points in 2021: 2 misses out of 70. Seats decided by
15 to 25 points: 23 misses out of 56.

The reason is arithmetic. A 7.6-point swing moves a two-party margin by 15.2
points, so 15.2 is the line where the baseline flips a seat. A seat held by 3
points flips under almost any positive swing, and local variation can't save
it. A seat held by 16 points sits right on the boundary, where the ±5 points
of local variation decides it either way.

So the errors track distance from the tipping point, not closeness.

In [ ]:
tipping = 2 * swing
dist = (seats["tmc_lead_2021"] - tipping).abs()
import pandas as pd
from src.models import uniform_swing_prediction, target
frame = pd.DataFrame({"dist": dist, "missed": (uniform_swing_prediction(seats) != target(seats)).values})
frame["band"] = pd.cut(frame["dist"], [0, 3, 6, 10, 20, 200], labels=["0-3", "3-6", "6-10", "10-20", "20+"])
frame.groupby("band", observed=True)["missed"].agg(["sum", "count", "mean"])

![miss rate](../figures/02_miss_by_tipping_distance.png)

## By region

In [ ]:
by_region

Kolkata and Howrah is the worst, 10 of 27 wrong. North Bengal the best, 4 of
54. Urban seats moved least like the state as a whole.

## Which features matter

In [ ]:
logistic_coefficients(seats).head(10)

In [ ]:
permutation_table(seats).head(8)

The 2021 margin and the region do most of the work, which is close to saying
the model has rediscovered the baseline and then added a regional correction
to it.

## The seats it gets wrong

In [ ]:
misses = baseline_misses(seats)
misses[["ac_name", "region", "winner_party_2021", "winner_party_2026",
        "tmc_lead_2021", "local_swing", "swing_vs_statewide"]].head(15)